In [1]:
import requests
import litellm
from bs4 import BeautifulSoup
import json, os
from tqdm import tqdm

In [2]:
html_file = "wildcard-episodes-start_to_20251004.html"
output_file = "extracted_urls.txt"

with open(html_file, "r", encoding="utf-8") as file:
    soup = BeautifulSoup(file, "html.parser")

urls_raw = [a.get("href") for a in soup.find_all("a", href=True)]
# filter only for URLs containing the string "/transcripts/", and remove duplicates
urls_raw = [url for url in urls_raw if "/transcripts/" in url]

seen = set()
urls = []
for item in urls_raw:
    if item not in seen:
        urls.append(item)
        seen.add(item)

with open(output_file, "w", encoding="utf-8") as outfile:
    for url in urls:
        outfile.write(f"{url}\n")

print(f"Extracted {len(urls)} URLs to {output_file}")

Extracted 76 URLs to extracted_urls.txt


In [3]:
# Set litellm to connect to the local Ollama instance
# NOTE: Ensure Ollama is running and the model you specify (e.g., llama3) is pulled.
# To pull the model: ollama pull llama3
#
# REQUIRED DEPENDENCIES: pip install requests litellm beautifulsoup4

# --- Configuration ---
# You can change the model to any model you have locally with Ollama (e.g., 'llama2', 'mistral')
LLM_MODEL = "ollama/granite3.3"
# LLM_MODEL = "ollama/gemma3:4b"


gemini = False
if gemini:
    LLM_MODEL = "gemini/gemini-2.5-flash-preview-05-20"
    # Provide your own key via the environment; never commit it to the repo.
    os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

In [4]:
def fetch_url_content(url: str) -> str | None:
    """
    Fetches the textual content from a given URL and extracts the main text body
    using the specified CSS selector.

    Args:
        url: The URL to fetch.

    Returns:
        The cleaned, relevant content as a string, or None if the fetch fails.
    """
    print(f"Fetching content from: {url}...")
    try:
        # Use a common User-Agent to avoid getting blocked
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        # Increase timeout for potentially slower network requests
        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)

        # Parse the raw HTML content
        raw_html = response.text
        soup = BeautifulSoup(raw_html, "html.parser")

        # Use the provided CSS selector to find the main transcript text
        # Selector: #main-section > article > div.transcript.storytext
        target_selector = "#main-section > article > div.transcript.storytext"
        target_element = soup.select_one(target_selector)

        if target_element:
            print(f"Successfully located content using selector: {target_selector}")
            # Extract and return the clean text, preserving line breaks for better readability
            return target_element.get_text(separator="\n", strip=True)
        else:
            print(
                f"Warning: Specific content selector '{target_selector}' not found. Falling back to extracting text from the entire body."
            )
            # Fallback: Extract text from the entire <body> tag as the next best clean option
            body_tag = soup.find("body")
            if body_tag:
                return body_tag.get_text(separator="\n", strip=True)

            # Last resort: return the raw HTML (handled by the length check in main)
            return raw_html

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during URL fetching: {e}")
        print(
            "Note: If running in a restricted environment, this fetch may fail. Using dummy content instead."
        )
        return None


def track_cost_callback(kwargs, completion_response, start_time, end_time):
    cost = kwargs["response_cost"]
    # print(f"Request cost: ${cost}")  # Send this data to your analytics/db


# litellm.success_callback = [track_cost_callback]


def ask_ollama(content: str, instruction: str, output_constraint=""):
    """
    Basic Ask Ollama interface

    Args:
        content: The text content fetched from the URL, used as context.
        question: The specific question to ask the LLM.
    """

    system_message = (
        "You are a specialized text analysis engine. Your sole task is to process the provided "
        "TRANSCRIPT and output ONLY the requested data in a single JSON object. "
        "Do not generate any conversational text, explanations, or filler. "
        f"Strictly adhere to the output format: {output_constraint}"
    )

    user_prompt = f"TRANSCRIPT:\n---\n{content}\n---\n\nINSTRUCTION: {instruction}"

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
    ]

    try:
        response = litellm.completion(
            model=LLM_MODEL,
            messages=messages,
            temperature=0.0,  # Low temperature for factual extraction
            stream=False,
            # response_format={
            #     "type": "json_object",
            #     "schema": json_schema,
            # },  # Force JSON output via schema
        )

        json_string = response.choices[0].message.content.strip()
        return json_string
    except litellm.exceptions.APIError as e:
        print(f"\n--- LLM Error ---")
        print(
            f"An API error occurred. Make sure Ollama is running and the model '{LLM_MODEL}' is downloaded."
        )
        print(f"Error details: {e}")
        print("--------------------")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")

    return None

In [5]:
print(urls[1])
transcript_text = fetch_url_content(urls[1])

https://www.npr.org/transcripts/nx-s1-5552701
Fetching content from: https://www.npr.org/transcripts/nx-s1-5552701...
Successfully located content using selector: #main-section > article > div.transcript.storytext


In [26]:
# Split transcript by \n, convert into dataframe, keep only rows that starts with "MARTIN"
import pandas as pd
df = pd.DataFrame({'row': transcript_text.split('\n')})
df = df[df['row'].str.startswith("MARTIN")]
df = df[df['row'].str.len() > 15]
df = df.reset_index(drop=True)
display(df)

,row
0,MARTIN: Makes you feel significant.
1,"MARTIN: I'm Rachel Martin, and this is WILD CA..."
2,"MARTIN: Each week, my guest answers questions ..."
3,MARTIN: I was thinking about what sets Matthew...
4,MARTIN: Hey. Yeah?
...,...
150,MARTIN: Just a moment you want to linger in a ...
151,MARTIN: Which one do you choose?
152,MARTIN: Matthew McConaughey. His new book is r...
153,"MARTIN: His new book is called (laughter) ""Poe..."


In [27]:
for i, r in tqdm(df.iterrows(), total=len(df)):
    # 1. Yes/No
    instruction_1 = "Is Martin asking someone to pick from one to three or one out of three cards? Output ONLY 'y' or 'n'."
    df.loc[i, "ask_to_pick"] = ask_ollama(
        r["row"], instruction_1, output_constraint="'y' or 'n'"
    )

    # 2. Categorization
    instruction_2 = "Identify the category explicitly mentioned by Martin. Valid categories: 'memory', 'insight', 'beliefs', or 'no category'. Output ONLY one of these words."
    df.loc[i, "category"] = ask_ollama(
        r["row"],
        instruction_2,
        output_constraint="One of: 'memory', 'insight', 'beliefs', 'no category'",
    )

    # 3. Extraction
    instruction_3 = "Extract the full, exact question asked in the transcript. If no question is present, output ONLY the word 'NONE'."
    df.loc[i, "question"] = ask_ollama(
        r["row"], instruction_3, output_constraint="Extracted question or 'NONE'"
    )

100%|██████████| 155/155 [11:05<00:00,  4.29s/it]


In [28]:
display(df)
df.to_clipboard()

,row,ask_to_pick,category,question
0,MARTIN: Makes you feel significant.,"{\n ""Is Martin asking someone to pick from on...","{\n ""category"": ""beliefs""\n}","{\n ""Extracted question"": ""NONE""\n}"
1,"MARTIN: I'm Rachel Martin, and this is WILD CA...","{\n ""Is Martin asking someone to pick from on...","""no category""","{\n ""Extracted question"": ""NONE""\n}"
2,"MARTIN: Each week, my guest answers questions ...","{\n ""Is Martin asking someone to pick from on...","""beliefs""","{\n ""Extracted question"": ""NONE""\n}"
3,MARTIN: I was thinking about what sets Matthew...,n,"{\n ""category"": ""insight""\n}","{\n ""Extracted question"": ""NONE""\n}"
4,MARTIN: Hey. Yeah?,"{\n ""Is Martin asking someone to pick from on...","""no category""","{\n ""Extracted question"": ""NONE""\n}"
...,...,...,...,...
150,MARTIN: Just a moment you want to linger in a ...,"{\n ""Is Martin asking someone to pick from on...","""no category""","{\n ""Extracted question"": ""NONE""\n}"
151,MARTIN: Which one do you choose?,"{\n ""Is_Martin_asking_for_one_out_of_three_op...","""no category""","{\n ""Extracted question"": ""Which one do you c..."
152,MARTIN: Matthew McConaughey. His new book is r...,"{\n ""Is Martin asking someone to pick from on...","{\n ""category"": ""no category""\n}","{\n ""Extracted question"": ""NONE""\n}"
153,"MARTIN: His new book is called (laughter) ""Poe...","{\n ""Is Martin asking someone to pick from on...","""no category""","{\n ""Extracted question"": ""NONE""\n}"


In [ ]:
qs to ask:

on category:
did martin say the word "round"? if yes then extract category

on card picking:
did martin announce a number in the beginning of the sentence? if yes then extract question

On question: just extract all the question